# 📊 Day 02: Monotonic Stack — The Next Greater Element Weapon

---

## What Makes a Stack "Monotonic"?

A monotonic stack is just a regular stack where you enforce an order — either always increasing or always decreasing from bottom to top.

Before pushing a new element, you **pop everything that violates the order**. Those popped elements have just "found" their answer (the new element is their next greater/smaller).

```
Monotonic Decreasing Stack (for "next greater"):

Process [2, 1, 2, 4, 3] left to right:

  Push 2:  stack = [2]
  Push 1:  stack = [2, 1]       (1 < 2, order maintained)
  Push 2:  pop 1 (next greater of 1 is 2), stack = [2, 2]
  Push 4:  pop 2 (next greater is 4), pop 2 (next greater is 4)
           stack = [4]
  Push 3:  stack = [4, 3]       (3 < 4, order maintained)
  Done:    4, 3 have no next greater → -1
```

**Every element is pushed once and popped once → O(n) total!**

---

## 🎯 Problem 1: Daily Temperatures (LC #739)

Given daily temperatures, for each day find how many days you have to wait until a warmer day. If no warmer day exists, answer is 0.

```
Input:  [73, 74, 75, 71, 69, 72, 76, 73]
Output: [1,  1,  4,  2,  1,  1,  0,  0]
        (73→74 is 1 day, 75→76 is 4 days, etc.)
```

Brute force would be O(n²) — checking every future day. Monotonic stack does it in O(n).

**The idea:** Stack holds indices of days waiting for a warmer day. When we find a warmer day, we pop all days that are cooler — they've found their answer.

In [ ]:
def dailyTemperatures(temperatures):
    """
    Monotonic decreasing stack of indices.
    Time: O(n), Space: O(n)
    """
    n = len(temperatures)
    result = [0] * n
    stack = []  # Stack of indices (temps in decreasing order)
    
    for i in range(n):
        # Pop all days that are cooler than today
        while stack and temperatures[i] > temperatures[stack[-1]]:
            prev_day = stack.pop()
            result[prev_day] = i - prev_day  # Days waited
        stack.append(i)
    
    # Remaining in stack have no warmer day → already 0
    return result

temps = [73, 74, 75, 71, 69, 72, 76, 73]
print(f"Temps:  {temps}")
print(f"Result: {dailyTemperatures(temps)}")  # [1,1,4,2,1,1,0,0]

# Let's trace it
print("\nTrace:")
stack = []
for i, t in enumerate(temps):
    popped = []
    while stack and t > temps[stack[-1]]:
        popped.append(stack.pop())
    stack.append(i)
    print(f"  Day {i} (temp={t}): popped={popped}, stack={stack}")

---

## 🎯 Problem 2: Next Greater Element I (LC #496)

Given `nums1` (subset of `nums2`), find the next greater element in `nums2` for each element of `nums1`.

```
nums1 = [4, 1, 2],  nums2 = [1, 3, 4, 2]
Output: [-1, 3, -1]

For 4: no greater element after 4 in nums2 → -1
For 1: next greater after 1 is 3 → 3
For 2: no greater element after 2 → -1
```

**Approach:** Process nums2 with a monotonic stack, build a map of `element → next_greater`. Then look up each element of nums1.

In [ ]:
def nextGreaterElement(nums1, nums2):
    """
    Build next-greater map from nums2, then look up nums1.
    Time: O(n + m), Space: O(n)
    """
    next_greater = {}  # element → its next greater element
    stack = []
    
    for num in nums2:
        # Pop elements smaller than num — they found their next greater
        while stack and num > stack[-1]:
            next_greater[stack.pop()] = num
        stack.append(num)
    
    # Everything left in stack has no next greater
    return [next_greater.get(num, -1) for num in nums1]

print(nextGreaterElement([4, 1, 2], [1, 3, 4, 2]))   # [-1, 3, -1]
print(nextGreaterElement([2, 4], [1, 2, 3, 4]))       # [3, -1]

---

## 🎯 Problem 3: Largest Rectangle in Histogram (LC #84)

This is one of the most elegant monotonic stack applications — and a hard problem that shows up a LOT.

```
Given heights of bars in a histogram, find the largest rectangle.

Input:  [2, 1, 5, 6, 2, 3]
Output: 10 (rectangle of height 5, width 2 using bars at index 2 and 3)
```

**Core insight:** For each bar, we want to know how far it can extend left and right (i.e., the first shorter bar on each side). A monotonic increasing stack gives us both boundaries.

When we pop a bar (because we hit a shorter bar), the popped bar's rectangle extends from the current stack top (+1) to the current index (-1).

In [ ]:
def largestRectangleArea(heights):
    """
    Monotonic increasing stack approach.
    Time: O(n), Space: O(n)
    """
    stack = []  # Stack of indices (heights in increasing order)
    max_area = 0
    
    for i, h in enumerate(heights):
        # Pop bars taller than current — they can't extend further right
        while stack and heights[stack[-1]] > h:
            height = heights[stack.pop()]
            # Width: from (stack top + 1) to (i - 1)
            width = i if not stack else i - stack[-1] - 1
            max_area = max(max_area, height * width)
        stack.append(i)
    
    # Process remaining bars in stack
    while stack:
        height = heights[stack.pop()]
        width = len(heights) if not stack else len(heights) - stack[-1] - 1
        max_area = max(max_area, height * width)
    
    return max_area

print(largestRectangleArea([2, 1, 5, 6, 2, 3]))  # 10
print(largestRectangleArea([2, 4]))                # 4
print(largestRectangleArea([1]))                   # 1

**Why is this problem important?** It's a building block for "Maximal Rectangle" (LC #85), which is the 2D version. Each row of the matrix becomes a histogram, and you solve it row by row.

---

## 🎯 Problem 4: Trapping Rain Water (LC #42)

Another legendary problem. Can be solved with monotonic stack, two pointers, or DP.

```
Input:  [0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]
Output: 6 units of water
```

**Two-pointer approach** (most elegant for this problem):

Walk from both ends. At each step, the shorter side determines the water level. Move the shorter pointer inward.

In [ ]:
def trap(height):
    """
    Two-pointer approach for trapping rain water.
    Time: O(n), Space: O(1)
    """
    left, right = 0, len(height) - 1
    left_max = right_max = 0
    water = 0
    
    while left < right:
        if height[left] < height[right]:
            # Left side is the bottleneck
            if height[left] >= left_max:
                left_max = height[left]  # Update max, no water here
            else:
                water += left_max - height[left]  # Water trapped!
            left += 1
        else:
            # Right side is the bottleneck
            if height[right] >= right_max:
                right_max = height[right]
            else:
                water += right_max - height[right]
            right -= 1
    
    return water

print(trap([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]))  # 6
print(trap([4, 2, 0, 3, 2, 5]))                       # 9

---

## 🗺️ Monotonic Stack Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  MONOTONIC STACK PATTERNS                                           ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. NEXT GREATER ELEMENT: Decreasing stack                          ║
║     Pop when current > stack top → popped found their next greater  ║
║                                                                      ║
║  2. NEXT SMALLER ELEMENT: Increasing stack                          ║
║     Pop when current < stack top → popped found their next smaller  ║
║                                                                      ║
║  3. HISTOGRAM RECTANGLE: Increasing stack of indices                ║
║     On pop: width = i − stack_top − 1, height = popped height      ║
║                                                                      ║
║  4. TRAPPING WATER: Two pointers or monotonic stack                 ║
║     Two pointers: O(1) space, move the shorter side inward          ║
║                                                                      ║
║  KEY: Every element pushed/popped at most once → O(n) total         ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Pattern |
|---|---------|------------|--------|
| 739 | Daily Temperatures | Medium | Next greater (indices) |
| 496 | Next Greater Element I | Easy | Next greater + hashmap |
| 503 | Next Greater Element II (circular) | Medium | Process array twice |
| 84 | Largest Rectangle in Histogram | Hard | Increasing stack |
| 42 | Trapping Rain Water | Hard | Two pointers / stack |
| 85 | Maximal Rectangle | Hard | Histogram per row |
| 901 | Online Stock Span | Medium | Next greater variant |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Monotonic stack: enforce order by popping before pushing        │
├──────────────────────────────────────────────────────────────────────┤
│  📌 "Next greater" = decreasing stack. "Next smaller" = increasing │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Each element pushed and popped at most once → O(n) guaranteed  │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Histogram rectangle: width calculation is the tricky part      │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Trapping water: two-pointer O(1) space is the cleanest        │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Expression Evaluation

Calculators, Reverse Polish Notation, and parsing nested expressions.

---

### 🎉 Monotonic Stack — Your O(n) Secret Weapon!
See "next greater/smaller"? Monotonic stack. Every time.